In [66]:
# =========================================================
# 0. IMPORTS
# =========================================================

import os
import io
import random
import warnings
import contextlib

warnings.filterwarnings("ignore")

# =========================================================
# GLOBAL SEED / DETERMINISM
# =========================================================

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# =========================================================
# CORE
# =========================================================

import joblib
import numpy as np
import pandas as pd

random.seed(SEED)
np.random.seed(SEED)

# =========================================================
# METRICS
# =========================================================

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# =========================================================
# MODELS
# =========================================================

import lightgbm as lgb
from xgboost import XGBRegressor

# =========================================================
# PREPROCESSING
# =========================================================

from sklearn.preprocessing import StandardScaler

# =========================================================
# NEURAL NETS
# =========================================================

import tensorflow as tf

tf.get_logger().setLevel("ERROR")
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2

In [ ]:
# =========================================================
# 1. GLOBAL SETTINGS
# =========================================================

# time resolution and target variable
time_resolution = "qh"   # "qh" or "h"
target_col = "id1"       # "id1" or "id3"

# models used in the comparison
selected_models = ["lgb", "xgb", "nn"]

# exported model: "lgb", "xgb", or "nn"
model_for_export = "nn"

# first year used as test set; all earlier years are training data
test_year = 2025

# lag settings
fundamental_lags = [1, 2]

# output paths
prediction_export_path = f"../data/{model_for_export}_predictions_{time_resolution}.csv"
artifact_export_path = f"../models/{model_for_export}_{test_year - 1}_model.pkl"

In [68]:
# =========================================================
# 2. PREDICTOR DEFINITIONS
# =========================================================

# current, non-lagged predictors used by the model
predictor_vars = [
    "day_ahead_price",
    "load_actual_mw",
    "load_delta",
    "wind_delta",
    "solar_delta",
    "import_delta",
    "export_delta",
    "net_import_total",
    "fossil_gas_mw",
    "biomass_mw",
    "hydro_run_of_river_and_poundage_mw",
    "hydro_water_reservoir_mw",
    "hydro_pumped_storage_mw",
    "solar_mw",
    "wind_onshore_mw",
    "outage_total_true",
    "ramp_outage",
]

# subset of predictors for which lagged versions are added
lagged_fundamental_vars = [
    "load_delta",
    "wind_delta",
    "solar_delta",
    "net_import_total",
    "ramp_outage",
]

time_features = [
    "hour_sin",
    "hour_cos",
    "month_sin",
    "month_cos",
    "free_day",
]

In [69]:
# =========================================================
# 3. HELPER FUNCTIONS
# =========================================================

@contextlib.contextmanager
def suppress_training_output():
    with contextlib.redirect_stdout(io.StringIO()):
        with contextlib.redirect_stderr(io.StringIO()):
            yield



def add_lag_features(df, columns, lags):
    df = df.copy()

    for col in columns:
        if col not in df.columns:
            continue

        for lag in lags:
            df[f"{col}_lag{lag}"] = df[col].shift(lag)

    return df


def build_feature_columns(df, predictor_vars, lagged_fundamental_vars, fundamental_lags, time_features):
    current_features = [col for col in predictor_vars if col in df.columns]

    lagged_features = [
        f"{col}_lag{lag}"
        for col in lagged_fundamental_vars
        for lag in fundamental_lags
        if f"{col}_lag{lag}" in df.columns
    ]

    available_time_features = [col for col in time_features if col in df.columns]

    feature_cols = current_features + lagged_features + available_time_features
    return list(dict.fromkeys(feature_cols))


def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def evaluate_price_metrics(actual_price, pred_price):
    residuals = np.asarray(actual_price) - np.asarray(pred_price)
    abs_residuals = np.abs(residuals)

    return {
        "ResidualMean": np.mean(residuals),
        "MAE": mean_absolute_error(actual_price, pred_price),
        "RMSE": rmse(actual_price, pred_price),
        "R2": r2_score(actual_price, pred_price),
        "p95_abs_residual": np.percentile(abs_residuals, 95),
        "p99_abs_residual": np.percentile(abs_residuals, 99),
    }


def make_random_validation_split(X_train, y_train, valid_fraction=0.15, seed=SEED):
    n_valid = int(np.floor(len(X_train) * valid_fraction))

    if n_valid < 1:
        raise ValueError("Validation set would be empty. Increase training data or valid_fraction.")

    rng = np.random.RandomState(seed)
    val_idx = rng.choice(X_train.index.to_numpy(), size=n_valid, replace=False)

    X_val = X_train.loc[val_idx].copy()
    y_val = y_train.loc[val_idx].copy()
    X_tr = X_train.drop(val_idx).copy()
    y_tr = y_train.drop(val_idx).copy()

    return X_tr, y_tr, X_val, y_val


def compute_mean_bias(y_true, y_pred):
    return float(np.mean(np.asarray(y_true) - np.asarray(y_pred)))


def apply_bias_correction(y_pred, bias_correction):
    return np.asarray(y_pred) + bias_correction


def format_metric_table(df_metrics):
    ordered_cols = [
        "Model",
        "ResidualMean",
        "MAE",
        "RMSE",
        "R2",
        "MeanError",
        "p95_abs_residual",
        "p99_abs_residual",
    ]

    existing_cols = [
        col for col in ordered_cols
        if col in df_metrics.columns
    ]

    return df_metrics[existing_cols].reset_index(drop=True)

In [70]:
# =========================================================
# 4. LOAD DATA
# =========================================================

df = pd.read_csv(
    f"../data/combined_data_{time_resolution}.csv",
    parse_dates=["datetime"],
)

df["datetime"] = pd.to_datetime(df["datetime"], utc=True).dt.tz_convert("Europe/Vienna")
df = df.sort_values("datetime").reset_index(drop=True)
df = df.loc[:, ~df.columns.duplicated()].copy()

print("data loaded:", df.shape)
print("start:", df["datetime"].min())
print("end:  ", df["datetime"].max())

data loaded: (140256, 132)
start: 2022-01-01 00:00:00+01:00
end:   2025-12-31 23:45:00+01:00


In [71]:
# =========================================================
# 5. CREATE TARGET AND LAGGED FEATURES
# =========================================================

spread_col = f"spread_{target_col}"
df[spread_col] = df[target_col] - df["day_ahead_price"]

df = add_lag_features(df, lagged_fundamental_vars, fundamental_lags)

print("target:", spread_col)
print("lags:  ", fundamental_lags)

target: spread_id1
lags:   [1, 2]


In [72]:
# =========================================================
# 6. BUILD FEATURE LIST
# =========================================================

feature_cols = build_feature_columns(
    df=df,
    predictor_vars=predictor_vars,
    lagged_fundamental_vars=lagged_fundamental_vars,
    fundamental_lags=fundamental_lags,
    time_features=time_features,
)

print(f"n_features = {len(feature_cols)}")
print(feature_cols)

n_features = 32
['day_ahead_price', 'load_actual_mw', 'load_delta', 'wind_delta', 'solar_delta', 'import_delta', 'export_delta', 'net_import_total', 'fossil_gas_mw', 'biomass_mw', 'hydro_run_of_river_and_poundage_mw', 'hydro_water_reservoir_mw', 'hydro_pumped_storage_mw', 'solar_mw', 'wind_onshore_mw', 'outage_total_true', 'ramp_outage', 'load_delta_lag1', 'load_delta_lag2', 'wind_delta_lag1', 'wind_delta_lag2', 'solar_delta_lag1', 'solar_delta_lag2', 'net_import_total_lag1', 'net_import_total_lag2', 'ramp_outage_lag1', 'ramp_outage_lag2', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'free_day']


In [73]:
# =========================================================
# 7. BUILD MODELING DATAFRAME AND TRAIN/TEST SPLIT
# =========================================================

needed_cols = ["datetime", target_col, "day_ahead_price", spread_col] + feature_cols
needed_cols = list(dict.fromkeys([col for col in needed_cols if col in df.columns]))

df_model = df[needed_cols].dropna().copy()

train_df = df_model[df_model["datetime"].dt.year < test_year].copy()
test_df = df_model[df_model["datetime"].dt.year == test_year].copy()

X_train = train_df[feature_cols].copy()
y_train = train_df[spread_col].copy()
X_test = test_df[feature_cols].copy()
y_test = test_df[spread_col].copy()

print("df_model shape:", df_model.shape)
print("train_df shape:", train_df.shape)
print("test_df shape: ", test_df.shape)
print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)
print("train period:", train_df["datetime"].min(), "->", train_df["datetime"].max())
print("test period: ", test_df["datetime"].min(), "->", test_df["datetime"].max())

df_model shape: (139772, 35)
train_df shape: (104828, 35)
test_df shape:  (34944, 35)
X_train: (104828, 32)
X_test:  (34944, 32)
train period: 2022-01-01 01:00:00+01:00 -> 2024-12-31 23:45:00+01:00
test period:  2025-01-01 00:00:00+01:00 -> 2025-12-31 23:45:00+01:00


In [74]:
# =========================================================
# 8. MODEL PIPELINES
# =========================================================

PARAM_SETS = {
    "A": {
        "lgb": {
            "objective": "mae",
            "n_estimators": 650,
            "learning_rate": 0.07,
            "max_depth": 10,
            "num_leaves": 95,
            "min_child_samples": 10,
            "subsample": 0.6,
            "colsample_bytree": 0.6,
            "reg_alpha": 0.01,
            "reg_lambda": 0.0,
        },
        "xgb": {
            "objective": "reg:absoluteerror",
            "eval_metric": "mae",
            "n_estimators": 500,
            "learning_rate": 0.05,
            "max_depth": 8,
            "min_child_weight": 1,
            "subsample": 0.7,
            "colsample_bytree": 1.0,
            "gamma": 0.3,
            "reg_alpha": 0.01,
        },
        "nn": {
            "units_1": 256,
            "units_2": 128,
            "dropout_1": 0.2,
            "dropout_2": 0.1,
            "l2_reg": 1e-6,
            "learning_rate": 0.001,
            "loss_name": "mae",
            "epochs": 70,
            "batch_size": 128,
        },
    },

    "B": {
        "lgb": {
            "objective": "mae",
            "n_estimators": 400,
            "learning_rate": 0.08,
            "max_depth": 6,
            "num_leaves": 31,
            "min_child_samples": 40,
            "subsample": 0.9,
            "colsample_bytree": 0.8,
            "reg_alpha": 10.0,
            "reg_lambda": 10.0,
        },
        "xgb": {
            "objective": "reg:absoluteerror",
            "eval_metric": "mae",
            "n_estimators": 400,
            "learning_rate": 0.08,
            "max_depth": 6,
            "min_child_weight": 4,
            "subsample": 0.8,
            "colsample_bytree": 0.9,
            "gamma": 0.5,
            "reg_alpha": 0.0,
            "reg_lambda": 10.0,
        },
        "nn": {
            "units_1": 128,
            "units_2": 64,
            "dropout_1": 0.25,
            "dropout_2": 0.15,
            "l2_reg": 1e-5,
            "learning_rate": 0.002,
            "loss_name": "mae",
            "epochs": 70,
            "batch_size": 128,
        },
    },
}


def get_params(model_name, param_set):
    return PARAM_SETS[param_set.upper()][model_name].copy()


def package_model_result(model, scaler, X_val, y_val, y_val_pred_raw, y_test, y_pred_raw):
    bias_correction = compute_mean_bias(y_val, y_val_pred_raw)
    y_val_pred = apply_bias_correction(y_val_pred_raw, bias_correction)
    y_pred = apply_bias_correction(y_pred_raw, bias_correction)

    return {
        "model": model,
        "scaler": scaler,
        "bias_correction": bias_correction,
        "val_index": X_val.index.to_numpy(),
        "val_pred": y_val_pred,
        "val_pred_raw": y_val_pred_raw,
        "test_pred": y_pred,
        "test_pred_raw": y_pred_raw,
        "test_residuals": y_test.to_numpy() - y_pred,
        "test_mae": mean_absolute_error(y_test, y_pred),
        "test_rmse": rmse(y_test, y_pred),
    }


def run_lgb_pipeline(X_train, y_train, X_test, y_test, seed=SEED, valid_fraction=0.15, param_set="B"):
    X_tr, y_tr, X_val, y_val = make_random_validation_split(
        X_train, y_train, valid_fraction=valid_fraction, seed=seed
    )

    params = get_params("lgb", param_set)

    model = lgb.LGBMRegressor(
        **params,
        random_state=seed,
        n_jobs=-1,
    )

    model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric="l1",
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
    )

    return package_model_result(
        model=model,
        scaler=None,
        X_val=X_val,
        y_val=y_val,
        y_val_pred_raw=model.predict(X_val),
        y_test=y_test,
        y_pred_raw=model.predict(X_test),
    )


def run_xgb_pipeline(X_train, y_train, X_test, y_test, seed=SEED, valid_fraction=0.15, param_set="B"):
    X_tr, y_tr, X_val, y_val = make_random_validation_split(
        X_train, y_train, valid_fraction=valid_fraction, seed=seed
    )

    params = get_params("xgb", param_set)

    model = XGBRegressor(
        **params,
        random_state=seed,
        n_jobs=-1,
        tree_method="hist",
    )

    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

    return package_model_result(
        model=model,
        scaler=None,
        X_val=X_val,
        y_val=y_val,
        y_val_pred_raw=model.predict(X_val),
        y_test=y_test,
        y_pred_raw=model.predict(X_test),
    )


def run_nn_pipeline(X_train, y_train, X_test, y_test, seed=SEED, valid_fraction=0.15, param_set="B"):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    tf.keras.backend.clear_session()

    X_tr, y_tr, X_val, y_val = make_random_validation_split(
        X_train, y_train, valid_fraction=valid_fraction, seed=seed
    )

    params = get_params("nn", param_set)

    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_val_s = scaler.transform(X_val)
    X_test_s = scaler.transform(X_test)

    model = Sequential([
        Input(shape=(X_tr_s.shape[1],)),
        Dense(
            params["units_1"],
            activation="relu",
            kernel_regularizer=l2(params["l2_reg"]),
        ),
        Dropout(params["dropout_1"]),
        Dense(
            params["units_2"],
            activation="relu",
            kernel_regularizer=l2(params["l2_reg"]),
        ),
        Dropout(params["dropout_2"]),
        Dense(1),
    ])

    model.compile(
        optimizer=Adam(learning_rate=params["learning_rate"]),
        loss=params["loss_name"],
    )

    callbacks = [
        EarlyStopping(
            monitor="val_loss",
            patience=8,
            restore_best_weights=True,
        )
    ]

    model.fit(
        X_tr_s,
        y_tr,
        validation_data=(X_val_s, y_val),
        epochs=params["epochs"],
        batch_size=params["batch_size"],
        verbose=0,
        callbacks=callbacks,
        shuffle=False,
    )

    return package_model_result(
        model=model,
        scaler=scaler,
        X_val=X_val,
        y_val=y_val,
        y_val_pred_raw=model.predict(X_val_s, verbose=0).ravel(),
        y_test=y_test,
        y_pred_raw=model.predict(X_test_s, verbose=0).ravel(),
    )

In [75]:
# =========================================================
# 9. TRAIN SELECTED MODELS
# =========================================================

pipeline_map = {
    "lgb": run_lgb_pipeline,
    "xgb": run_xgb_pipeline,
    "nn": run_nn_pipeline,
}

results = {}

for model_name in selected_models:
    if model_name not in pipeline_map:
        raise ValueError(f"Unsupported model: {model_name}")

    print(f"running {model_name.upper()}")

    with suppress_training_output():
        res = pipeline_map[model_name](X_train, y_train, X_test, y_test)

    val_df = train_df.loc[res["val_index"]].copy()

    res["feature_cols"] = feature_cols
    res["test_actual_price"] = test_df[target_col].to_numpy()
    res["test_da_price"] = test_df["day_ahead_price"].to_numpy()
    res["test_datetime"] = test_df["datetime"].to_numpy()
    res["val_actual_price"] = val_df[target_col].to_numpy()
    res["val_da_price"] = val_df["day_ahead_price"].to_numpy()
    res["val_datetime"] = val_df["datetime"].to_numpy()

    results[model_name] = res

running LGB
running XGB
running NN


In [76]:
# =========================================================
# 10. COMPARE VALIDATION AND TEST METRICS
# =========================================================

model_order = ["lgb", "xgb", "nn"]

validation_rows = []
test_rows = []

first_res = next(iter(results.values()))

validation_rows.append({
    "Model": "naive",
    **evaluate_price_metrics(
        first_res["val_actual_price"],
        first_res["val_da_price"],
    ),
})

test_rows.append({
    "Model": "naive",
    **evaluate_price_metrics(
        first_res["test_actual_price"],
        first_res["test_da_price"],
    ),
})


for model_name in model_order:

    if model_name not in results:
        continue

    res = results[model_name]

    validation_rows.append({
        "Model": model_name,
        **evaluate_price_metrics(
            res["val_actual_price"],
            res["val_da_price"] + res["val_pred"],
        ),
    })

    test_rows.append({
        "Model": model_name,
        **evaluate_price_metrics(
            res["test_actual_price"],
            res["test_da_price"] + res["test_pred"],
        ),
    })


validation_df = format_metric_table(
    pd.DataFrame(validation_rows)
)

test_df_metrics = format_metric_table(
    pd.DataFrame(test_rows)
)


print("\n" + "=" * 100)
print("VALIDATION METRICS")
print("=" * 100)
print(validation_df.to_string(index=False))


print("\n\n")

print("=" * 100)
print(f"TEST METRICS ({test_year})")
print("=" * 100)
print(test_df_metrics.to_string(index=False))


VALIDATION METRICS
Model  ResidualMean       MAE      RMSE       R2  p95_abs_residual  p99_abs_residual
naive -6.514806e+00 41.832473 71.551851 0.707340        128.514000        273.916000
  lgb -2.892059e-16 33.789958 61.168391 0.786117        104.193351        223.455608
  xgb  1.399331e-07 33.130590 59.735991 0.796017        102.833421        220.560310
   nn -4.550696e-08 35.169901 62.196387 0.778867        106.917188        223.505077



TEST METRICS (2025)
Model  ResidualMean       MAE      RMSE       R2  p95_abs_residual  p99_abs_residual
naive      0.082428 31.226034 66.303141 0.356545         91.608500        195.340900
  lgb      4.957476 28.690032 63.276522 0.413949         77.366462        190.126900
  xgb      5.750228 29.153434 63.891902 0.402495         79.407899        193.794902
   nn      2.410782 28.042926 62.506563 0.428125         77.485414        180.422117


In [82]:
# =========================================================
# 11. EXPORT PREDICTIONS
# =========================================================

if model_for_export not in results:
    raise ValueError(f"model_for_export must be one of {list(results)}")

best_res = results[model_for_export]

prediction_export_df = pd.DataFrame({
    "datetime": best_res["test_datetime"],
    target_col.upper(): best_res["test_actual_price"],
    f"{target_col.upper()}_hat": best_res["test_da_price"] + best_res["test_pred"],
})

prediction_export_df = pd.concat(
    [
        prediction_export_df,
        test_df[best_res["feature_cols"]].reset_index(drop=True),
    ],
    axis=1,
)

prediction_export_df.to_csv(prediction_export_path, index=False)

print("saved:", prediction_export_path)

saved: ../data/xgb_predictions_qh.csv


In [83]:
# =========================================================
# 12. EXPORT MODEL ARTIFACT
# =========================================================

model_artifact = {
    "model": best_res["model"],
    "scaler": best_res["scaler"],
    "bias_correction": best_res["bias_correction"],
    "feature_cols": best_res["feature_cols"],
    "target_col": target_col,
    "spread_col": spread_col,
    "time_resolution": time_resolution,
    "test_year": test_year,
}

joblib.dump(model_artifact, artifact_export_path)

print("saved:", artifact_export_path)

saved: ../models/xgb_2024_model.pkl
